In [ ]:
# from delta import *
# from pyspark.sql import SparkSession

# builder = SparkSession.builder \
#     .appName("RetailPipeline") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [1]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip

    builder = SparkSession.builder \
        .appName("retail_pipeline") \
        .master("local[*]") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

ERROR StatusLogger Reconfiguration failed: No configuration found for '5c29bfd' at 'null' in 'null'
ERROR StatusLogger Reconfiguration failed: No configuration found for 'Default' at 'null' in 'null'
26/04/22 14:12:08 WARN Utils: Your hostname, YTuSurface5 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/22 14:12:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/yiqingtu/spark/spark-3.5.5-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/yiqingtu/.ivy2/cache
The jars for the packages stored in: /home/yiqingtu/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4846a444-a3da-487a-9cc1-61781a9bd81c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 177ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

In [7]:
bronze_orders = "../delta/bronze/orders"
df_bronze = spark.read.load(bronze_orders)

# need panda library
df_bronze.limit(1000).toPandas()


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,subcategory,product_name,sales,quantity,discount,profit
0,764,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001990,Office Supplies,Envelopes,Staple envelope,11.36,2.000,0.0,5.3392
1,765,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-EN-10001532,Office Supplies,Envelopes,Brown Kraft Recycled Envelopes,50.94,3.000,0.0,25.4700
2,766,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,TEC-AC-10003174,Technology,Accessories,Plantronics S12 Corded Telephone Headset System,646.74,6.000,0.0,258.6960
3,767,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-BI-10004187,Office Supplies,Binders,3-ring staple pack,5.64,3.000,0.0,2.7072
4,768,CA-2014-162775,2014-01-13,2014-01-15,Second Class,CS-12250,Chris Selesnick,Corporate,United States,Bossier City,...,71111,South,OFF-ST-10000025,Office Supplies,Storage,Fellowes Stor/Drawer Steel Plus Storage Drawers,572.58,6.000,0.0,34.3548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,7981,CA-2014-103800,2014-01-03,2014-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"""Message Book, Wirebound, Four 5 1/2"""" X 4"""" F...","200 Dupl. Sets/Book""",16.448,2.0,0.2000
62,6475,CA-2014-149524,2014-01-14,2014-01-15,First Class,BS-11590,Brendan Sweed,Corporate,United States,Philadelphia,...,19140,East,FUR-BO-10003433,Furniture,Bookcases,Sauder Cornerstone Collection Library,61.96,4.000,0.5,-53.2856
63,6475,CA-2014-149524,2014-01-14,2014-01-15,First Class,BS-11590,Brendan Sweed,Corporate,United States,Philadelphia,...,19140,East,FUR-BO-10003433,Furniture,Bookcases,Sauder Cornerstone Collection Library,61.96,4.000,0.5,-53.2856
64,717,CA-2014-130092,2014-01-11,2014-01-14,First Class,SV-20365,Seth Vernon,Consumer,United States,Dover,...,19901,East,FUR-FU-10000010,Furniture,Furnishings,"DAX Value U-Channel Document Frames, Easel Back",9.94,2.000,0.0,3.0814


In [8]:
df_bronze.show()

+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+--------------+--------------+-----------+------+---------------+---------------+-----------+--------------------+-------+--------+--------+-------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|    customer_name|  segment|      country|          city|         state|postal_code|region|     product_id|       category|subcategory|        product_name|  sales|quantity|discount| profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+--------------+--------------+-----------+------+---------------+---------------+-----------+--------------------+-------+--------+--------+-------+
|   764|CA-2014-162775|2014-01-13|2014-01-15|  Second Class|   CS-12250|  Chris Selesnick|Corporate|United States|  Bossier City|     Louisiana|      71111| South|OFF-EN-10001990|Office Supplies|  Envelopes|    